# NimbusPay ticket triage: fine-tuning a small model (VIMA3YA take-home)

This notebook is the whole pipeline, top to bottom, on a free Colab T4:

| Part | What happens |
|---|---|
| A | Audit `train.jsonl`, clean it, write `cleaning_log.csv` |
| B | Two baselines on dev (fixed prompt, and rules-in-prompt as a reference) |
| C | LoRA fine-tuning pipeline (chat template, loss mask, sequence length) |
| D | Controlled experiments, one change at a time |
| E | Error analysis on dev with the final model |
| Final | Adapter in PEFT format + `predictions_test.jsonl` |

**How to run:** Runtime → Change runtime type → T4 GPU, then Run all. Put `training_assessment_1.zip` (or the unzipped `candidate_pack/` folder) in `/content` or in the Drive folder set in the config cell.
A fresh run trains only the final model (`FINAL` in the config cell). The other experiment rows are read from `experiments_log.jsonl`; set `RUN_ALL_EXPERIMENTS = True` to re-train all of them.

AI use: I used an AI assistant (Claude) throughout this work. Details are in `report.md`.

**Note on the saved outputs:** they are from my full run with `RUN_ALL_EXPERIMENTS = True` (all six training runs, close to 3 hours on a T4). I set the flag back to `False` afterwards without re-running, so a fresh Run all trains only the final model and finishes in about 1 hour 50 minutes.

In [1]:
!pip install -q peft accelerate bitsandbytes
!pip uninstall -y -q torchao

In [2]:
import os, sys, json, re, csv, time, math, random, gc, glob, shutil, zipfile, collections, datetime as dt
import numpy as np

# ---------------- config ----------------
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # 1.54B params, Apache-2.0, not gated, has a system role
SEED = 0
FINAL = "clean_aug"                # which experiment config (Part D) is the final model
RUN_BASELINES = True
RUN_ALL_EXPERIMENTS = False       # False: train only FINAL, read the other rows from experiments_log.jsonl
USE_DRIVE = True                   # keep adapters / logs / predictions on Google Drive so a disconnect loses nothing
DRIVE_DIR = "/content/drive/MyDrive/nimbuspay_triage"

EFFECTIVE_BATCH = 16               # examples per optimizer step
MAX_TOKENS_PER_MICROBATCH = 4096   # padded tokens per forward pass (memory limit, see Part C)
GEN_BATCH = 16                     # batch size for dev generation during experiments
Counter = collections.Counter

In [3]:
WORK_DIR = "/content/work"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        WORK_DIR = DRIVE_DIR
    except Exception as e:
        print("Drive not mounted, using /content/work:", e)
os.makedirs(WORK_DIR, exist_ok=True)

def find_pack():
    roots = ["/content", WORK_DIR, "."]
    for root in roots:
        for p in [f"{root}/candidate_pack"] + glob.glob(f"{root}/*/candidate_pack"):
            if os.path.exists(f"{p}/data/train.jsonl"): return p
    for root in roots:
        for z in glob.glob(f"{root}/*training_assessment*.zip"):
            zipfile.ZipFile(z).extractall("/content")
            return "/content/candidate_pack"
    raise FileNotFoundError("Upload training_assessment_1.zip (or candidate_pack/) to /content or to " + WORK_DIR)

PACK_DIR = find_pack()
DATA_DIR = f"{PACK_DIR}/data"
sys.path.insert(0, PACK_DIR)
from score import score, KEYS          # the official scorer

def load_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]
def save_jsonl(rows, p):
    with open(p, "w", encoding="utf-8") as f:
        for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")

train_raw = load_jsonl(f"{DATA_DIR}/train.jsonl")
dev_rows = load_jsonl(f"{DATA_DIR}/dev.jsonl")
test_rows = load_jsonl(f"{DATA_DIR}/test_inputs.jsonl")
SCHEMA_TEXT = open(f"{PACK_DIR}/SCHEMA.md", encoding="utf-8").read()
U = lambda r: r["messages"][1]["content"]      # ticket text
A = lambda r: r["messages"][2]["content"]      # label string
print(PACK_DIR, "| train", len(train_raw), "dev", len(dev_rows), "test", len(test_rows), "| work dir:", WORK_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/candidate_pack | train 1445 dev 200 test 400 | work dir: /content/drive/MyDrive/nimbuspay_triage


# Part A: Audit the training data

Plan: look at the structure first, then duplicates and leakage, then check every label against the rules in `SCHEMA.md`.
Each check prints the evidence. The decisions are collected in one `clean_train()` function at the end, which writes `cleaning_log.csv`.

## A1. Structure of the rows and of the label strings

In [4]:
for name, rows in [("train", train_raw), ("dev", dev_rows), ("test", test_rows)]:
    print(f"{name:5} rows={len(rows)} unique ids={len({r['id'] for r in rows})}",
          "roles=", dict(Counter(tuple(m["role"] for m in r["messages"]) for r in rows)),
          "system prompts=", len({r["messages"][0]["content"] for r in rows}))

def label_problem(a):
    try: o = json.loads(a)
    except json.JSONDecodeError: return "not valid JSON"
    if list(o) != KEYS: return "keys: " + ",".join(o)
    return "ok"

for name, rows in [("train", train_raw), ("dev", dev_rows)]:
    print("\n" + name, "label strings:")
    for k, v in Counter(label_problem(A(r)) for r in rows).most_common(): print(f"  {v:5d}  {k}")

print("\nexamples of labels that are not valid JSON (cut off in the middle):")
for r in [r for r in train_raw if label_problem(A(r)) == "not valid JSON"][:3]: print("  ", r["id"], A(r))
print("\nexamples of labels with the wrong keys:")
legacy = [r for r in train_raw if label_problem(A(r)).startswith("keys")]
for r in legacy[:3]: print("  ", r["id"], A(r))
print("\nvalues of 'prio' in those rows:", dict(Counter(json.loads(A(r))["prio"] for r in legacy)))
print("their Source:", dict(Counter(re.search(r"Source: (\S+)", U(r)).group(1) for r in legacy)),
      "| all train rows by Source:", dict(Counter(re.search(r"Source: (\S+)", U(r)).group(1) for r in train_raw)))
print("date formats in those rows:", dict(Counter(re.sub(r"\d", "9", json.loads(A(r))["txn_date"] or "null") for r in legacy)))

train rows=1445 unique ids=1445 roles= {('system', 'user', 'assistant'): 1445} system prompts= 1
dev   rows=200 unique ids=200 roles= {('system', 'user', 'assistant'): 200} system prompts= 1
test  rows=400 unique ids=400 roles= {('system', 'user'): 400} system prompts= 1

train label strings:
   1350  ok
     70  keys: category,prio,amount_paise,txn_date,txn_id,channel,language,needs_human
     25  not valid JSON

dev label strings:
    200  ok

examples of labels that are not valid JSON (cut off in the middle):
   tr-00060 {"category": "kyc", "priority": "P3", "amount_paise": null, "txn_date": null, "t
   tr-00095 {"category": "other", "priority": "P3", "amount_paise": null, "txn_date": null, "txn_id": null, "c
   tr-00125 {"category": "payment_failed", "priority": "P3", "amount_paise": 392500, "txn_date": "2026-08-12", "t

examples of labels with the wrong keys:
   tr-00020 {"category": "refund", "prio": "low", "amount_paise": 97800, "txn_date": "02/09/2026", "txn_id": null, "channel

**Findings**

1. **25 labels are cut off mid-string** and do not parse. A model trained on them learns to stop early and emit broken JSON, which scores zero on every field. → drop.
2. **70 rows use an older label format**: key `prio` with `low/medium/high` instead of `priority` with `P3/P2/P1`, and `DD/MM/YYYY` dates. All 70 come from `ivr-transcript`, so it looks like one source that was never migrated to spec v2. The content is recoverable, so I convert them (and check after conversion that they agree with the v2 rules, see A5).

## A2. Duplicates and dev/test leakage

In [5]:
def split_ticket(u):
    """-> (received date, source, customer's own text with quoted earlier mails removed)"""
    head, body = u.split("\n\n", 1)
    rec = dt.date.fromisoformat(re.search(r"Received: (\d{4}-\d\d-\d\d)", head).group(1))
    src = re.search(r"Source: (\S+)", head).group(1)
    live = "\n".join(l for l in body.split("\n") if not l.lstrip().startswith(">"))
    return rec, src, live

def norm_key(u):
    """received date + ticket text with case, spacing, punctuation and a trailing 'thx' removed"""
    rec, _, t = split_ticket(u)
    t = re.sub(r"\b(thx|thanks)\b\W*$", "", t.lower().strip())
    return rec.isoformat() + "|" + re.sub(r"[^a-z0-9₹]", "", t)

exact = Counter(U(r) for r in train_raw)
print("train rows whose exact text appears more than once:", sum(v for v in exact.values() if v > 1), "in", sum(v > 1 for v in exact.values()), "groups")
groups = collections.defaultdict(list)
for r in train_raw: groups[norm_key(U(r))].append(r)
near = [g for g in groups.values() if len(g) > 1]
print("after normalising whitespace/punctuation:", sum(len(g) for g in near), "rows in", len(near), "groups;",
      "groups with conflicting labels:", sum(len({A(r) for r in g}) > 1 for g in near))
g = next(g for g in near if len({U(r) for r in g}) > 1)
print("\nexample of a near-duplicate pair:")
for r in g: print("  ", r["id"], repr(U(r)[-110:]))

dev_text, dev_keys = {U(r) for r in dev_rows}, {norm_key(U(r)) for r in dev_rows}
test_text, test_keys = {U(r) for r in test_rows}, {norm_key(U(r)) for r in test_rows}
print("\ntrain rows identical to a dev row:", sum(U(r) in dev_text for r in train_raw),
      "| near-identical:", sum(norm_key(U(r)) in dev_keys for r in train_raw))
print("train rows identical to a test row:", sum(U(r) in test_text for r in train_raw),
      "| near-identical:", sum(norm_key(U(r)) in test_keys for r in train_raw))

train rows whose exact text appears more than once: 120 in 60 groups
after normalising whitespace/punctuation: 195 rows in 97 groups; groups with conflicting labels: 0

example of a near-duplicate pair:
   tr-00009 't was for Rs 4,694.75.  Mode: debit card.  This is the third time I am contacting you.  Please look into it.  '
   tr-00473 '40. It was for Rs 4,694.75. Mode: debit card. This is the third time I am contacting you. Please look into it.'

train rows identical to a dev row: 30 | near-identical: 45
train rows identical to a test row: 0 | near-identical: 0


**Findings**

3. **Duplicates.** 60 exact pairs, and 97 groups once I ignore double spaces and a tacked-on "Thx" (the near-duplicates are the same ticket with the same Received date, only spaced differently). Labels inside a group never conflict, so I keep the first copy and drop the rest. Leaving them in only over-weights those tickets.
4. **Dev leakage.** 30 train rows are byte-identical to dev rows and 15 more are whitespace/"Thx" variants of dev rows. The assignment says never train on dev rows, and with them in, the dev score would be inflated. → drop all 45. No overlap with test.

## A3. Junk tickets

In [6]:
def is_junk(u):
    """no real message: fewer than 3 distinct words after removing the wrappers"""
    t = split_ticket(u)[2]
    t = re.sub(r"\[IVR transcript\]|@NimbusPayCare|^Subject:.*$", "", t, flags=re.M)
    words = re.findall(r"[A-Za-z]+", t)
    return len(set(w.lower() for w in words)) < 3

junk = [r for r in train_raw if is_junk(U(r))]
print("junk rows in train:", len(junk), "| dev:", sum(is_junk(U(r)) for r in dev_rows), "| test:", sum(is_junk(U(r)) for r in test_rows))
for r in junk: print("  ", r["id"], repr(U(r).split("\n\n", 1)[1][:30]), "->", A(r)[:120])

junk rows in train: 10 | dev: 0 | test: 0
   tr-00034 'test test 123' -> {"category": "refund", "priority": "P2", "amount_paise": 1474500, "txn_date": "2026-08-20", "txn_id": "NP6537766118", "c
   tr-00164 'test' -> {"category": "offers", "priority": "P3", "amount_paise": 6500, "txn_date": null, "txn_id": null, "channel": null, "langu
   tr-00318 '' -> {"category": "kyc", "priority": "P3", "amount_paise": null, "txn_date": null, "txn_id": null, "channel": null, "language
   tr-00555 'test test 123' -> {"category": "payment_failed", "priority": "P2", "amount_paise": 1913400, "txn_date": "2026-06-15", "txn_id": "NP7243953
   tr-00790 'asdf asdf' -> {"category": "refund", "priority": "P3", "amount_paise": 472700, "txn_date": "2026-08-18", "txn_id": "NP1363195589", "ch
   tr-00794 'test test 123' -> {"category": "payment_failed", "priority": "P3", "amount_paise": 51400, "txn_date": null, "txn_id": null, "channel": "ne
   tr-01032 'test' -> {"category": "refund", "priority": "P3", "amount_p

5. **Junk rows**: `test`, `test test 123`, `asdf asdf` or an empty body, each with a full, confident label (amounts, transaction ids, dates that appear nowhere in the text). These teach the model to invent values. → drop.

## A4. A rule checker built from SCHEMA.md

Most fields are a deterministic function of the ticket once the category is known. So I wrote the spec as code and compare it with the labels. Where they disagree, either my checker is wrong or the label is.

To know which, I first run the checker on **dev**, whose labels are guaranteed to follow the spec. Dev is used here only as a measuring stick for the checker (same role as `score.py`). No dev text or label goes into training.

The checker does not decide `category` or `language` (those are judged in A6). It takes the labelled category as given and checks the other six fields.

In [7]:
TXN_CATS = {"payment_failed", "refund", "fraud", "offers"}        # the others must have null transaction fields
ID_RE = r"(?<![A-Za-z0-9])N[ \-]?P((?:[ \-]{0,2}\d){10})(?!\d)"   # NP + 10 digits, any case, spaces/hyphens allowed
# legal threat (spec list) and "already contacted support" phrases (spec examples + the phrasings found in train)
LEGAL = re.compile(r"\b(rbi|ombudsman|consumer court|consumer forum|lawyer|legal notice|legal action|legal|court)\b", re.I)
REPEAT = re.compile(r"(third time|second time|\d(st|nd|rd|th) time|again and again|baar baar|bar bar|contacted support about this before"
                    r"|teesri baar|doosri baar|dusri baar|pehle bhi|second complaint|third complaint"
                    r"|already (raised|complained|called|wrote|written|reported|contacted))", re.I)

def rule_txn_id(t):
    m = re.search(ID_RE, t, re.I)
    return "NP" + re.sub(r"\D", "", m.group(1)) if m else None

def rule_channels(t):
    """payment modes named in the text ('scratch card', also when misspelt, is a reward and not a payment mode)"""
    s = re.sub(r"\bs[a-z]{6} card", lambda m: "scratchcard" if sorted(m.group(0)[:7]) == sorted("scratch") else m.group(0), t.lower())
    pats = [(r"\bupi\b", "upi"), (r"net ?banking|internet banking", "netbanking"), (r"\bwallet\b", "wallet"), (r"\b(card|visa|mastercard)\b", "card")]
    return [v for p, v in pats if re.search(p, s)]

NUM = r"(\d[\d,]*(?:\.\d+)?)"
AMT = re.compile(r"(?:₹|rs\.?|inr)\s*" + NUM + r"(?:\s*(k|lakhs?|lac|lacs)\b)?(?:/-)?|" + NUM + r"\s*(k|lakhs?|lacs?|rupees|rupaye|rupay|rs\.?|inr|/-)(?![a-z])", re.I)
def amt_paise(m):
    n = m.group(1) or m.group(3); unit = (m.group(2) or m.group(4) or "").lower()
    v = float(n.replace(",", ""))
    if unit == "k": v *= 1000
    elif unit.startswith("la"): v *= 100000
    return int(round(v * 100))

def scrub(t):
    """hide numbers that are neither amounts nor dates: transaction ids, app versions, ticket numbers, bank references"""
    t = re.sub(ID_RE, " TXNID ", t, flags=re.I)
    t = re.sub(r"\b\d+\.\d+\.\d+\b", " VER ", t)
    t = re.sub(r"#\d+", " TICKET ", t)
    return re.sub(r"\b\d{11,}\b", " BANKREF ", t)

BEFORE = re.compile(r"balance|limit|\bfees?\b", re.I); AFTER = re.compile(r"pade hain|balance|limit|\bfees?\b", re.I)
def rule_amounts(t):
    """amounts in the text, in paise, skipping balances / limits / fees (spec: 'ignore other amounts')"""
    t = scrub(t); out = []
    for m in AMT.finditer(t):
        start = max(t.rfind(". ", 0, m.start()), t.rfind("\n", 0, m.start()), m.start() - 40, 0)
        if BEFORE.search(t[start:m.start()]) or AFTER.search(t[m.end():m.end() + 14]): continue
        out.append(amt_paise(m))
    return out

MONTH = {m: i + 1 for i, m in enumerate("jan feb mar apr may jun jul aug sep oct nov dec".split())}
def rule_dates(t, rec):
    """every date the customer's own text could mean, as ISO strings"""
    out = []; s = scrub(t).lower()
    def add(d, mo, y=None):
        try:
            if y is None:                                   # no year: most recent such date on or before Received
                x = dt.date(rec.year, mo, d)
                if x > rec: x = dt.date(rec.year - 1, mo, d)
            else: x = dt.date(y if y > 99 else 2000 + y, mo, d)
            out.append(x.isoformat())
        except ValueError: pass
    mon = "(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*"
    for m in re.finditer(r"\b(\d{4})-(\d\d)-(\d\d)\b", s): add(int(m[3]), int(m[2]), int(m[1]))
    for m in re.finditer(r"\b(\d{1,2})[/\-.](\d{1,2})[/\-.](\d{4}|\d{2})\b", s): add(int(m[1]), int(m[2]), int(m[3]))
    for m in re.finditer(r"\b(\d{1,2})(?:st|nd|rd|th)?\s+" + mon + r"(?:,?\s+(\d{4}))?", s): add(int(m[1]), MONTH[m[2]], int(m[3]) if m[3] else None)
    for m in re.finditer(r"\b" + mon + r"\s+(\d{1,2})(?:st|nd|rd|th)?\b(?:,?\s+(\d{4}))?", s): add(int(m[2]), MONTH[m[1]], int(m[3]) if m[3] else None)
    for m in re.finditer(r"\b(\d+)\s+(?:days? ago|din pehle)", s): out.append((rec - dt.timedelta(int(m[1]))).isoformat())
    for pat, n in [(r"day before yesterday|\bparso\b", 2), (r"(?<!before )\byesterday\b|\bkal\b", 1), (r"\btoday\b|\baaj\b", 0)]:
        if re.search(pat, s): out.append((rec - dt.timedelta(n)).isoformat())
    return out

def rule_priority(cat, amount, t):
    if cat == "fraud" or (amount is not None and amount >= 50000 * 100) or LEGAL.search(t): return "P1"
    if (cat in ("payment_failed", "refund") and amount is not None and amount >= 5000 * 100) or cat == "account_access": return "P2"
    return "P3"

def rule_needs_human(prio, t):
    return prio == "P1" or bool(REPEAT.search(t))

def check(u, o):
    """label o vs spec rules -> list of (field, label value, what the rules allow). Empty list = consistent."""
    rec, _, t = split_ticket(u); cat = o["category"]; out = []
    if cat not in TXN_CATS:
        out += [(k, o[k], None) for k in ("amount_paise", "txn_date", "txn_id", "channel") if o[k] is not None]
    else:
        tid = rule_txn_id(t)
        if tid != o["txn_id"]: out.append(("txn_id", o["txn_id"], tid))
        for field, allowed in [("channel", rule_channels(t)), ("amount_paise", rule_amounts(t)), ("txn_date", rule_dates(t, rec))]:
            if (o[field] not in allowed) if allowed else (o[field] is not None): out.append((field, o[field], allowed))
    p = rule_priority(cat, o["amount_paise"], t)
    if p != o["priority"]: out.append(("priority", o["priority"], p))
    nh = rule_needs_human(o["priority"], t)
    if nh != o["needs_human"]: out.append(("needs_human", o["needs_human"], nh))
    return out

dev_bad = [(r["id"], check(U(r), json.loads(A(r)))) for r in dev_rows]
dev_bad = [x for x in dev_bad if x[1]]
print(f"checker vs dev labels: {len(dev_bad)} of {len(dev_rows)} rows disagree")
for x in dev_bad[:5]: print("  ", x)

checker vs dev labels: 0 of 200 rows disagree


The checker agrees with all 200 dev labels on the six fields it covers, so a disagreement on train is very likely a label problem.

## A5. Checker on train

In [8]:
PRIO_MAP = {"low": "P3", "medium": "P2", "high": "P1"}
def convert_legacy(o):
    """old format -> v2: 'prio' low/medium/high -> 'priority' P3/P2/P1, DD/MM/YYYY -> YYYY-MM-DD"""
    o = dict(o); o["priority"] = PRIO_MAP[o.pop("prio")]
    if o.get("txn_date") and re.fullmatch(r"\d\d/\d\d/\d{4}", o["txn_date"]):
        d, m, y = o["txn_date"].split("/"); o["txn_date"] = f"{y}-{m}-{d}"
    return o

disagree = collections.defaultdict(list); n_checked = n_legacy_bad = 0
for r in train_raw:
    if label_problem(A(r)) == "not valid JSON" or is_junk(U(r)): continue
    o = json.loads(A(r)); was_legacy = "prio" in o
    if was_legacy: o = convert_legacy(o)
    n_checked += 1
    probs = check(U(r), o); n_legacy_bad += bool(probs) and was_legacy
    for f, lab, rule in probs: disagree[f].append((r["id"], lab, rule, split_ticket(U(r))[2].replace("\n", " | ")[:170]))
print("rows checked:", n_checked, "| converted legacy rows that still disagree with v2 rules:", n_legacy_bad)
for f, items in disagree.items():
    print(f"\n{f}: {len(items)} disagreements   (id, label, rule, text)")
    for it in items[:5]: print("   ", it)

flips = Counter(f"{lab}->{rule}" for _, lab, rule, _ in disagree["priority"])
print("\npriority label -> rule value:", dict(flips))
bad_id = [x for x in disagree["txn_id"] if x[2] is not None]
print("txn_id labels that are right but not normalised:", len(bad_id), "| e.g.", [x[1] for x in bad_id[:4]])
rupee = [x for x in disagree["amount_paise"] if len(set(x[2])) == 1 and x[1] in (x[2][0] // 100, round(x[2][0] / 100))]
print("amount labels equal to the rupee value (not multiplied by 100):", len(rupee), "of", len(disagree["amount_paise"]))

rows checked: 1410 | converted legacy rows that still disagree with v2 rules: 0

priority: 65 disagreements   (id, label, rule, text)
    ('tr-00013', 'P1', 'P3', 'Subject: Issue with my account |  | Hi, | KYC mein address update karna hai. Android app use kar raha hoon. Bank ref 782509120860 hai. Pehle bhi complaint kar chuka hoon.')
    ('tr-00018', 'P1', 'P3', 'Subject: Issue with my account |  | Dear NimbusPay support, | Referral bonus was not received after my friend signed up. Mode: UPI. My phone is a OnePlus Nord. The transa')
    ('tr-00023', 'P1', 'P3', 'URGENT!! How can you assist on verifying my identity? I am on the iOS app. Please look into it.')
    ('tr-00045', 'P3', 'P1', 'Subject: Help needed |  |  | I AM MISSING CASH FROM MY BANK ACCOUNT ACCORDING TO MY APP. The amount was INR 11,594. I am on the iOS app. Waiting for your reply. |  | Rega')
    ('tr-00059', 'P1', 'P3', '@NimbusPayCare A refund for a recent return has not arrived on my card. The amount was 3.7k. App ve

**Findings** (counts above include duplicate/leaked rows; the cleaning log has the final per-row list)

6. **Amount left in rupees.** The label holds the rupee value instead of paise (`Rs. 674` → `674`, `2.2 lakh` → `220000`, and `Rs 594.50` → `594`, so the fraction is lost too). → fix by recomputing from the text, only when the text has exactly one candidate amount. Several of these rows also show up under priority, because my priority rule was reading the wrong (rupee) amount. Once the amount is fixed those disagreements go away.
7. **Wrong priority.** About 60 rows have a priority the rules cannot produce from their own category, amount and text: `kyc` questions marked `P1`, refunds of ₹46,000 marked `P3`, fraud marked `P3`. It goes in both directions and I found no pattern by source, wording ("URGENT!!" does not explain it) or id range, so I treat it as random label noise. The `needs_human` disagreements printed above are these same rows: my check derives `needs_human` from the labelled priority, and in these rows `needs_human` matches the *correct* priority. So only the priority value is damaged, and after fixing priority no `needs_human` label needs to change. Priority is fully determined by the spec → fix by rule.
8. **Transaction ids not normalised** in the label (`np 81633 47892`, `NP-9718716109`). The spec wants `NP` + 10 digits. → fix.
9. The converted legacy rows agree with the v2 rules, so converting them (instead of dropping 70 IVR rows) is safe.

## A6. Category and language (not covered by the checker)

For these two I have no exact rule, so I look for labels that disagree with what the rest of the training set says: a cross-validated TF-IDF classifier for category and a small Hinglish word list for language.

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict
from sklearn.pipeline import make_pipeline, make_union

ok = [r for r in train_raw if label_problem(A(r)) != "not valid JSON" and not is_junk(U(r))]
X = [scrub(split_ticket(U(r))[2]) for r in ok]; y = [json.loads(A(r))["category"] for r in ok]
clf = make_pipeline(make_union(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True),
                               TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True)),
                    LogisticRegression(C=20, max_iter=3000))
proba = cross_val_predict(clf, X, y, cv=5, method="predict_proba")
classes = sorted(set(y)); pred = [classes[i] for i in proba.argmax(1)]
sus = sorted([(proba[i].max(), ok[i]["id"], y[i], pred[i], X[i].replace("\n", " | ")[:150]) for i in range(len(y)) if pred[i] != y[i]], reverse=True)
print(f"5-fold agreement with the category labels: {np.mean([a == b for a, b in zip(pred, y)]):.1%}; {len(sus)} disagreements. Most confident ones:")
for s in sus[:12]: print("  %.2f" % s[0], *s[1:])

HINGLISH = re.compile(r"\b(hai|hain|nahi|kar|karo|karna|karun|kiya|ka|ki|ke|ko|se|mein|hua|tha|thi|ho|hoon|raha|rahi|paise|mera|meri|kya|kaise|aur|lekin|abhi|bhi|gaya|gaye|wapas|baar|din|pehle|chahiye|bhai|namaste|dhanyavaad|jaldi|rupaye|rakam)\b", re.I)
lang_bad = [(r["id"], json.loads(A(r))["language"], split_ticket(U(r))[2][:90]) for r in ok
            if (json.loads(A(r))["language"] == "hi-en") != (len(HINGLISH.findall(split_ticket(U(r))[2])) >= 3)]
print(f"\nlanguage label vs word list: {len(lang_bad)} disagreements")
for x in lang_bad: print("  ", x)

5-fold agreement with the category labels: 97.5%; 35 disagreements. Most confident ones:
  0.90 tr-00257 fraud refund Subject: Help needed |  | Good morning, | I see a payment on my statement for a place I didn't make a purchase at. It was from the NimbusPay wallet. I
  0.82 tr-00729 refund payment_failed Hi, My card was declined but it shows as multiple charges? Rs. 2728 is the amount. It took place today.
  0.76 tr-01074 fraud payment_failed Subject: Complaint |  |  | It is urgency. Please freeze my card now. As someone in some odd remote town has withdrawn some money by using my card. As 
  0.74 tr-01443 fraud payment_failed [IVR transcript] Hi, Payment I did not make. Dated Jun 6. Total ₹439. Mode: internet banking. Please help.
  0.74 tr-01357 fraud payment_failed [IVR transcript] Hi, Payment I did not make. Dated Jun 6. Total ₹439. Mode: internet banking. Please help.
  0.71 tr-01286 account_access other [IVR transcript] Hi team, My card's frozen, what can I do? Please look into 

**Result: no label noise found in `category` or `language`.** I read the classifier's disagreements: they are cases where the simple classifier is wrong and the label is right (mostly `fraud`, which wins over refund/payment_failed by the spec and has the fewest examples). The few language disagreements are short Hinglish tickets that my word list undercounts. So I change nothing for these two fields. I am reporting this because I looked and found nothing, not because I skipped it.

## A7. What the model can and cannot learn from this data

The test model sees no rules, so each rule must be visible in the training examples. I counted how often each spec rule is exercised in train (after removing broken rows).

In [10]:
valid = []
for r in train_raw:
    if label_problem(A(r)) == "not valid JSON" or is_junk(U(r)): continue
    o = json.loads(A(r)); valid.append((r, convert_legacy(o) if "prio" in o else o))

def cover(name, f):
    n = sum(bool(f(split_ticket(U(r))[2], o, r)) for r, o in valid)
    print(f"  {n:5d}  ({n / len(valid):5.1%})  {name}")
print("spec rule or pattern -> train rows that exercise it")
cover("amount written with 'lakh'", lambda t, o, r: re.search(r"\blakhs?\b|\blacs?\b", t, re.I))
cover("amount written with 'k'", lambda t, o, r: re.search(r"\d\s?k\b", t))
cover("relative date: 'kal'", lambda t, o, r: re.search(r"\bkal\b", t, re.I))
cover("relative date: 'parso' / 'day before yesterday'", lambda t, o, r: re.search(r"\bparso\b|day before yesterday", t, re.I))
cover("relative date: 'N days ago' / 'N din pehle'", lambda t, o, r: re.search(r"days? ago|din pehle", t, re.I))
cover("date without year that resolves to the previous year", lambda t, o, r: o["txn_date"] and o["txn_date"][:4] != str(split_ticket(U(r))[0].year))
cover("second amount to ignore (balance / limit / fee)", lambda t, o, r: re.search(r"balance|limit|\bfees?\b|pade hain", t, re.I))
cover("'fee' specifically", lambda t, o, r: re.search(r"\bfees?\b", t, re.I) and o["category"] in TXN_CATS)
cover("other reference number (ticket / bank ref / order)", lambda t, o, r: re.search(r"#\d+|bank ref|order (no|number|id)", t, re.I))
cover("txn id written with spaces / hyphen / lower case", lambda t, o, r: o["txn_id"] and o["txn_id"] not in t)
cover("quoted earlier e-mail in the ticket", lambda t, o, r: "\n>" in U(r))
cover("legal threat", lambda t, o, r: LEGAL.search(t))
cover("account_access/kyc/other ticket that mentions an amount, date or NP id", lambda t, o, r: o["category"] not in TXN_CATS and (rule_amounts(t) or rule_dates(t, split_ticket(U(r))[0]) or rule_txn_id(t)))

spec rule or pattern -> train rows that exercise it
     27  ( 1.9%)  amount written with 'lakh'
     90  ( 6.4%)  amount written with 'k'
     13  ( 0.9%)  relative date: 'kal'
     40  ( 2.8%)  relative date: 'parso' / 'day before yesterday'
     67  ( 4.8%)  relative date: 'N days ago' / 'N din pehle'
      0  ( 0.0%)  date without year that resolves to the previous year
     81  ( 5.7%)  second amount to ignore (balance / limit / fee)
      0  ( 0.0%)  'fee' specifically
    152  (10.8%)  other reference number (ticket / bank ref / order)
    345  (24.5%)  txn id written with spaces / hyphen / lower case
     40  ( 2.8%)  quoted earlier e-mail in the ticket
     90  ( 6.4%)  legal threat
      0  ( 0.0%)  account_access/kyc/other ticket that mentions an amount, date or NP id


**Findings**

10. **Coverage gaps (not label errors, but they matter).** Some rules in the spec have few examples: `kal` (13 rows), `parso`/day before yesterday (40), `lakh` (27). Three have **zero** examples: a fee as the amount to ignore, a date without a year that falls into the previous year, and an `account_access`/`kyc`/`other` ticket that mentions transaction details (the spec says the four transaction fields are *always* null there). A model that never sees these cannot learn them, and the hidden test is said to be harder than dev. I use this for my own experiment in Part D (spec-driven augmentation).
11. **Very long tickets.** About 3% of rows are e-mails with 10 to 16 quoted support replies (around 8,000 characters). They contain dates that must be ignored. I keep them; they decide the sequence length (Part C).
12. Typos in the customer text (`pamyent`, `Srcatch card`) are present in train, dev and test alike. That is input noise the model has to live with, not a labelling problem → keep.

## A8. Cleaning

In [11]:
def dump_label(o):
    return json.dumps({k: o[k] for k in KEYS}, ensure_ascii=False)      # same serialisation as the dev labels

def clean_train(train_rows, dev_rows):
    """-> (clean rows, log of (id, action, reason)). Drops rows that cannot be trusted, fixes what the spec determines."""
    log, out, seen = [], [], {}
    dev_keys = {norm_key(U(r)) for r in dev_rows}
    for r in train_rows:
        rid, u = r["id"], U(r)
        # ---- problems with the whole row: drop
        if label_problem(A(r)) == "not valid JSON":
            log.append((rid, "drop", "label is cut off / not valid JSON")); continue
        if is_junk(u):
            log.append((rid, "drop", "junk or empty ticket text (e.g. 'test', 'asdf') with an arbitrary label")); continue
        k = norm_key(u)
        if k in dev_keys:
            log.append((rid, "drop", "same ticket as a dev row (exact or whitespace/'Thx' variant): leakage")); continue
        if k in seen:
            log.append((rid, "drop", f"duplicate of {seen[k]} (exact or whitespace/'Thx' variant)")); continue
        # ---- problems with single fields: fix
        o = json.loads(A(r)); why = []
        if "prio" in o:
            o = convert_legacy(o); why.append("legacy label format (key 'prio' low/medium/high, DD/MM/YYYY date) converted to spec v2")
        _, _, t = split_ticket(u)
        if o["txn_id"] is not None and not re.fullmatch(r"NP\d{10}", o["txn_id"]):
            fixed = "NP" + re.sub(r"\D", "", o["txn_id"])
            if fixed == rule_txn_id(t):
                why.append(f"txn_id not normalised ('{o['txn_id']}' -> '{fixed}')"); o["txn_id"] = fixed
        if o["category"] in TXN_CATS and o["amount_paise"] is not None:
            c = set(rule_amounts(t))
            if len(c) == 1 and o["amount_paise"] not in c and o["amount_paise"] in (min(c) // 100, round(min(c) / 100)):
                why.append(f"amount left in rupees instead of paise ({o['amount_paise']} -> {min(c)})"); o["amount_paise"] = min(c)
        p = rule_priority(o["category"], o["amount_paise"], t)
        if p != o["priority"]:
            why.append(f"priority contradicts the spec rules for this category/amount/text ({o['priority']} -> {p})"); o["priority"] = p
        nh = rule_needs_human(o["priority"], t)
        if nh != o["needs_human"]:
            why.append(f"needs_human contradicts the spec ({str(o['needs_human']).lower()} -> {str(nh).lower()})"); o["needs_human"] = nh
        left = check(u, o)
        if left:        # anything I cannot explain: do not guess, drop
            log.append((rid, "drop", "label still disagrees with the spec after fixes: " + "; ".join(f"{f}={l!r} vs rules {v!r}" for f, l, v in left))); continue
        seen[k] = rid
        if why: log.append((rid, "fix", "; ".join(why)))
        out.append({"id": rid, "messages": [r["messages"][0], r["messages"][1], {"role": "assistant", "content": dump_label(o)}]})
    return out, log

train_clean, cleaning_log = clean_train(train_raw, dev_rows)
with open(f"{WORK_DIR}/cleaning_log.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["id", "action", "reason"]); w.writerows(cleaning_log)
save_jsonl(train_clean, f"{WORK_DIR}/train_clean.jsonl")

print(f"train: {len(train_raw)} -> {len(train_clean)} rows | log:", dict(Counter(a for _, a, _ in cleaning_log)))
summary = Counter()
for _, action, why in cleaning_log:
    for part in why.split("; "): summary[action + ": " + re.sub(r" \(.*| of tr-\d+.*|:.*", "", part)] += 1
for k, v in summary.most_common(): print(f"  {v:4d}  {k}")
assert not any(check(U(r), json.loads(A(r))) for r in train_clean) and not {norm_key(U(r)) for r in train_clean} & dev_keys
print("every remaining row is consistent with the rule checker and none matches a dev row")
print("category mix after cleaning:", dict(Counter(json.loads(A(r))["category"] for r in train_clean)))

train: 1445 -> 1267 rows | log: {'drop': 178, 'fix': 175}
    98  drop: duplicate
    70  fix: legacy label format
    60  fix: priority contradicts the spec rules for this category/amount/text
    45  drop: same ticket as a dev row
    25  drop: label is cut off / not valid JSON
    25  fix: amount left in rupees instead of paise
    20  fix: txn_id not normalised
    10  drop: junk or empty ticket text
every remaining row is consistent with the rule checker and none matches a dev row
category mix after cleaning: {'payment_failed': 372, 'account_access': 162, 'refund': 280, 'fraud': 76, 'kyc': 153, 'offers': 133, 'other': 91}


## A9. Spec-driven augmentation (used as my own experiment in Part D)

For the coverage gaps in A7 I generate extra rows. Rules I set for myself:

* every new row is an **edit of a cleaned train row** (insert one sentence, or rewrite the amount / transaction id), so it uses only `train.jsonl` and `SCHEMA.md`;
* the edit is chosen so that **the spec fixes the new label** (for example "inserted *Yeh parso hua.*" → `txn_date` = Received − 2 days);
* a new row is kept only if the rule checker from A4 agrees with its label.

I did not look at the test tickets to design these. The list comes from the spec rules that A7 showed as rare or missing.

In [12]:
MONTHS = "Jan Feb Mar Apr May Jun Jul Aug Sep Oct Nov Dec".split()

def insert_sentence(u, sent, rng):
    """add one sentence inside the customer's own paragraph (never in the header, signature or quoted mail)"""
    head, body = u.split("\n\n", 1)
    lines = body.split("\n")
    stop = next((i for i, l in enumerate(lines) if l.startswith("Regards") or l.startswith(">")), len(lines))
    idx = max(range(stop), key=lambda i: (not lines[i].startswith("Subject:")) * len(lines[i]))
    parts = re.split(r"(?<=[.!?]) +(?=[A-Z0-9₹])", lines[idx])
    parts.insert(rng.randint(1, len(parts)), sent)
    lines[idx] = " ".join(parts)
    return head + "\n\n" + "\n".join(lines)

def fmt_rupees(x, rng):
    s = f"{x:,}" if rng.random() < 0.5 else str(x)
    return rng.choice(["Rs {}", "Rs. {}", "₹{}", "INR {}", "{} rupees"]).format(s)

def fmt_messy_id(digits, rng):
    """NP + 10 digits the way customers type it: any case, spaces or hyphens anywhere"""
    prefix = rng.choice(["NP", "np", "Np", "N P", "n p", "NP-", "np-", "NP ", "np "])
    chunks, i = [], 0
    while i < 10:
        n = min(rng.choice([2, 3, 4, 5, 10]), 10 - i); chunks.append(digits[i:i + n]); i += n
    return prefix + rng.choice([" ", "-", " "]).join(chunks)

def augment(clean_rows, n_per_type=45, seed=0):
    rng = random.Random(seed)
    pool = [(r, U(r), json.loads(A(r)), split_ticket(U(r))) for r in clean_rows if len(U(r)) < 1500]
    out = []
    def take(kind, cond, edit, n):
        cands = [p for p in pool if cond(p[2])]; rng.shuffle(cands); k = 0
        for r, u, o, sp in cands:
            if k >= n: break
            res = edit(u, dict(o), sp)
            if res is None or check(res[0], res[1]): continue          # checker must agree with the new label
            out.append({"id": f"aug-{len(out) + 1:05d}", "aug": kind, "source_id": r["id"],
                        "messages": [r["messages"][0], {"role": "user", "content": res[0]}, {"role": "assistant", "content": dump_label(res[1])}]})
            k += 1
    txn = lambda o: o["category"] in TXN_CATS
    hi = lambda o: o["language"] == "hi-en"
    digits = lambda k: "".join(rng.choice("0123456789") for _ in range(k))

    # 1. relative dates, resolved against Received
    def rel_date(u, o, sp):
        s, n = rng.choice([("Yeh kal hua.", 1), ("Kal ki baat hai.", 1), ("Yeh parso hua.", 2), ("Parso ki baat hai.", 2), ("Yeh {n} din pehle hua.", None), ("Aaj ki baat hai.", 0)] if hi(o)
                          else [("This happened yesterday.", 1), ("It was day before yesterday.", 2), ("That was day before yesterday.", 2), ("It took place {n} days ago.", None), ("It was today.", 0)])
        if n is None: n = rng.randint(3, 9); s = s.format(n=n)
        o["txn_date"] = (sp[0] - dt.timedelta(n)).isoformat()
        return insert_sentence(u, s, rng), o
    take("relative_date", lambda o: txn(o) and o["txn_date"] is None, rel_date, n_per_type)

    # 2. date without a year that would be in the future -> previous year
    def rollover(u, o, sp):
        rec = sp[0]; m = rng.randint(rec.month, 12); d = rng.randint(1, 28)
        if dt.date(rec.year, m, d) <= rec: return None
        s = rng.choice(["Yeh {d} {mon} ko hua.", "Transaction date {mon} {d} hai."] if hi(o)
                       else ["It took place on {d} {mon}.", "Transaction date {mon} {d}.", "Dated {d} {mon}."]).format(d=d, mon=MONTHS[m - 1])
        o["txn_date"] = dt.date(rec.year - 1, m, d).isoformat()
        return insert_sentence(u, s, rng), o
    take("year_rollover", lambda o: txn(o) and o["txn_date"] is None, rollover, n_per_type // 2)

    # 3. amount rewritten with k / lakh; priority and needs_human follow from the spec
    def k_lakh(u, o, sp):
        head, body = u.split("\n\n", 1)
        ms = [m for m in AMT.finditer(body) if amt_paise(m) == o["amount_paise"]]
        if len(ms) != 1 or len(rule_amounts(sp[2])) != 1: return None
        if rng.random() < 0.65:
            v = rng.choice([0.5, 0.6, 0.75, 0.9, 1, 1.2, 1.5, 1.75, 2, 2.5, 3.2, 4, 5.5]); paise = int(round(v * 100000 * 100))
            expr = rng.choice(["{} lakh", "{} lakhs", "Rs {} lakh", "₹{} lakh"]).format(f"{v:g}")
        else:
            v = rng.choice([1.2, 2.5, 4.9, 5, 5.5, 8, 12.5, 30, 49.9, 50, 62, 75]); paise = int(round(v * 1000 * 100))
            expr = rng.choice(["{}k", "Rs {}k", "₹{}k"]).format(f"{v:g}")
        contacted_before = o["needs_human"]            # source row is not P1, so needs_human there means "contacted before"
        o["amount_paise"] = paise
        o["priority"] = "P1" if paise >= 5000000 else ("P2" if o["category"] in ("payment_failed", "refund") and paise >= 500000 else "P3")
        o["needs_human"] = o["priority"] == "P1" or contacted_before
        return head + "\n\n" + body[:ms[0].start()] + expr + body[ms[0].end():], o
    take("k_lakh_amount", lambda o: txn(o) and o["priority"] != "P1" and o["amount_paise"] is not None, k_lakh, n_per_type)

    # 4. transaction id typed with odd case / spaces / hyphens (label stays the normalised id)
    def messy_id(u, o, sp):
        ms = list(re.finditer(r"(?<![A-Za-z0-9])N[ \-]?P(?:[ \-]{0,2}\d){10}(?!\d)", u, re.I))
        if len(ms) != 1: return None
        return u[:ms[0].start()] + fmt_messy_id(o["txn_id"][2:], rng) + u[ms[0].end():], o
    take("messy_txn_id", lambda o: o["txn_id"] is not None, messy_id, n_per_type)

    # 5. an amount to ignore: balance, limit, fee (often large enough to change priority if the model used it)
    def distract_amount(u, o, sp):
        x = rng.choice([rng.randint(50, 4000), rng.randint(5000, 49000), rng.randint(50000, 400000)])
        fee = rng.choice([10, 15, 25, 49, 99, 150])
        s = rng.choice(["Account mein abhi {a} pade hain.", "Meri daily limit {a} hai.", "{f} ki fees bhi lagi."] if hi(o)
                       else ["My account balance is {a} right now.", "FYI my daily limit is {a}.", "A fee of {f} was also charged.", "The convenience fee was {f}."])
        return insert_sentence(u, s.format(a=fmt_rupees(x, rng), f=fmt_rupees(fee, rng)), rng), o
    take("distractor_amount", txn, distract_amount, n_per_type)

    # 6. reference numbers that are not NimbusPay transaction ids
    def other_ref(u, o, sp):
        s = rng.choice(["Order number OD{a} hai.", "Bank ref {b} hai.", "Pichla ticket number #{c} tha."] if hi(o)
                       else ["My order number is OD{a}.", "Bank reference number is {b}.", "My earlier ticket number is #{c}.", "The UTR is {b}.", "Order id {a}."])
        return insert_sentence(u, s.format(a=digits(8), b=digits(12), c=digits(5)), rng), o
    take("other_reference", lambda o: txn(o) and o["priority"] != "P1" and not o["needs_human"], other_ref, n_per_type)

    # 7. account_access / kyc / other ticket that mentions transaction details: the four fields stay null.
    #    Amounts are kept under Rs 5,000 so I do not have to guess how a large amount would affect priority here.
    def non_txn(u, o, sp):
        opts = (["Yeh kal hua.", "Yeh {n} din pehle hua.", "Amount {a} tha.", "Transaction ID {i} hai.", "UPI se hua tha.", "Card se hua tha."] if hi(o)
                else ["This happened yesterday.", "It took place {n} days ago.", "The amount was {a}.", "The txn id is {i}.", "Mode: UPI.", "It was a card transaction.", "It was from the NimbusPay wallet."])
        for s in rng.sample(opts, rng.choice([1, 2])):
            u = insert_sentence(u, s.format(n=rng.randint(2, 9), a=fmt_rupees(rng.randint(50, 4500), rng), i="NP" + digits(10)), rng)
        return u, o
    take("non_txn_with_details", lambda o: not txn(o), non_txn, n_per_type)
    return out

train_aug = augment(train_clean, seed=SEED)
save_jsonl(train_aug, f"{WORK_DIR}/train_aug.jsonl")
print(len(train_aug), "augmented rows:", dict(Counter(r["aug"] for r in train_aug)))
shown = set()
for r in train_aug:
    if r["aug"] not in shown:
        shown.add(r["aug"]); print(f"\n[{r['aug']}] from {r['source_id']}\n  " + U(r).split("\n\n", 1)[1].replace("\n", " | ")[:330] + "\n  -> " + A(r))

292 augmented rows: {'relative_date': 45, 'year_rollover': 22, 'k_lakh_amount': 45, 'messy_txn_id': 45, 'distractor_amount': 45, 'other_reference': 45, 'non_txn_with_details': 45}

[relative_date] from tr-01370
  Hello, I did not make this payment that I see in the App. It was today. FYI my daily limit is 410 rupees. INR 25,183 is the amount. My phone is a Redmi Note 12. Mode: internet banking.
  -> {"category": "fraud", "priority": "P1", "amount_paise": 2518300, "txn_date": "2026-09-18", "txn_id": null, "channel": "netbanking", "language": "en", "needs_human": true}

[year_rollover] from tr-00088
  Subject: Issue with my account |  | Dear NimbusPay support, | My scratch card reward has not been added. It was a UPI transaction. Please look into it. It took place on 1 Sep. |  | Regards, | Kavya Reddy
  -> {"category": "offers", "priority": "P3", "amount_paise": null, "txn_date": "2025-09-01", "txn_id": null, "channel": "upi", "language": "en", "needs_human": false}

[k_lakh_amount] from

# Part C (first half): tokenizer, chat template, loss mask, sequence length

I do this before the baselines because the baselines use the same prompt formatting and generation code.

**Stack:** plain Hugging Face `transformers` + `peft` (+ `bitsandbytes` for the 4-bit run) with a short hand-written PyTorch training loop. I chose this over Unsloth/TRL because the assignment asks about the chat template, which tokens carry loss, and sequence length. With a hand-written loop each of those is a few visible lines that I can show and change live, and nothing depends on a trainer's defaults.

**Model:** `Qwen/Qwen2.5-1.5B-Instruct`. 1.54B parameters (limit 4B), Apache-2.0 and not gated, ChatML template with a real system role (so the fixed system prompt goes in unchanged), a multilingual tokenizer that handles Hinglish and splits numbers into single digits (useful for ids, dates and paise). It is small enough that a training run takes minutes on a T4, which is what lets me run six controlled experiments inside the free quota, and both 16-bit and 4-bit fit in memory so that comparison is fair.

In [13]:
import torch, torch.nn.functional as F, transformers, peft
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, GenerationConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__, "|", torch.cuda.get_device_name(0))

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
tok.padding_side = "left"      # only used for batched generation; training batches are padded by hand on the right
assert tok.pad_token_id is not None

def prompt_text(messages, system=None):
    """exactly system + user through the model's own chat template, ending where the assistant starts"""
    msgs = [{"role": "system", "content": system or messages[0]["content"]}, messages[1]]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def answer_text(messages):
    """what the template puts after the prompt for the assistant turn: the JSON followed by the end-of-turn token"""
    full = tok.apply_chat_template(messages[:3], tokenize=False)
    p = prompt_text(messages)
    assert full.startswith(p)
    return full[len(p):].rstrip("\n")

def encode(messages):
    """prompt and answer are tokenised separately, so the prompt tokens are identical to what the model sees at inference.
    labels = -100 on every prompt token: loss only on the JSON and the end-of-turn token."""
    p = tok(prompt_text(messages), add_special_tokens=False)["input_ids"]
    a = tok(answer_text(messages), add_special_tokens=False)["input_ids"]
    return {"input_ids": p + a, "labels": [-100] * len(p) + a}

ex = train_clean[0]["messages"]
print(prompt_text(ex) + "‖" + answer_text(ex))
e = encode(ex)
stop_ids = GenerationConfig.from_pretrained(BASE_MODEL).eos_token_id
stop_ids = stop_ids if isinstance(stop_ids, list) else [stop_ids]
print("\ntokens:", len(e["input_ids"]), "| tokens with loss:", sum(l != -100 for l in e["labels"]))
print("text that carries loss:", repr(tok.decode([l for l in e["labels"] if l != -100])))
assert e["labels"][-1] in stop_ids, "the last trained token must be a stop token, otherwise generation would not stop after the JSON"
print("base model generation defaults:", GenerationConfig.from_pretrained(BASE_MODEL).to_diff_dict())

torch 2.11.0+cu130 | transformers 5.18.0 | peft 0.21.1 | Tesla T4
<|im_start|>system
You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON.<|im_end|>
<|im_start|>user
Received: 2026-07-25 (Sat)
Source: app-chat

Hi, I have a top up that didn't go through. Why? It was today. Mode: internet banking. My phone is a Redmi Note 12. It is 4.7k in total. My account balance is 258 rupees right now. The txn id is NP1599046332. Waiting for your reply.<|im_end|>
<|im_start|>assistant
‖{"category": "payment_failed", "priority": "P3", "amount_paise": 470000, "txn_date": "2026-07-25", "txn_id": "NP1599046332", "channel": "netbanking", "language": "en", "needs_human": false}<|im_end|>

tokens: 214 | tokens with loss: 81
text that carries loss: '{"category": "payment_failed", "priority": "P3", "amount_paise": 470000, "txn_date": "2026-07-25", "txn_id": "NP1599046332", "channel": "netbanking", "language": "en", "needs_human": false}<|im_end|>'
base model generation defaults

**Chat template.** The prompt is built by `tok.apply_chat_template([system, user], add_generation_prompt=True)`, the same call used at inference, so training and test prompts are token-identical. Qwen has a system role, so no workaround is needed.

**Loss on the right tokens.** Loss is computed only on the assistant's JSON plus `<|im_end|>`. Prompt tokens have label −100. Training on the ticket text would spend capacity on modelling customers' typos, and leaving out `<|im_end|>` would produce a model that keeps talking after the JSON, which the strict scorer counts as invalid.

### Sequence length

In [14]:
def token_lengths(rows, with_answer):
    return np.array([len(tok(prompt_text(r["messages"]), add_special_tokens=False)["input_ids"])
                     + (len(tok(answer_text(r["messages"]), add_special_tokens=False)["input_ids"]) if with_answer else 0) for r in rows])
len_train = token_lengths(train_clean + train_aug, True)
for name, L in [("train (prompt+answer)", len_train), ("dev (prompt)", token_lengths(dev_rows, False)), ("test (prompt)", token_lengths(test_rows, False))]:
    print(f"{name:22} median {int(np.median(L)):5d}  p95 {int(np.percentile(L, 95)):5d}  p99 {int(np.percentile(L, 99)):5d}  max {L.max():5d}  rows > 512: {(L > 512).sum()}")
ans = np.array([len(tok(answer_text(r["messages"]), add_special_tokens=False)["input_ids"]) for r in train_clean])
print("answer length in tokens: median", int(np.median(ans)), "max", ans.max(), "(the inference limit is 200 new tokens)")
assert len_train.max() <= MAX_TOKENS_PER_MICROBATCH and ans.max() < 200

train (prompt+answer)  median   171  p95   226  p99  1753  max  2214  rows > 512: 36
dev (prompt)           median   106  p95   142  p99  1464  max  2115  rows > 512: 3
test (prompt)          median   108  p95   151  p99  1827  max  2089  rows > 512: 11
answer length in tokens: median 65 max 84 (the inference limit is 200 new tokens)


**Decision: no truncation.** Almost every ticket is under ~200 tokens, but about 3% are e-mails with a long quoted thread, a few thousand tokens each (numbers above). I keep them whole because:

* truncating from the right would cut off the label, and truncating the ticket would hide the exact thing the model has to learn there: the quoted replies contain dates that are *not* the transaction date;
* at test time the model is given the full ticket (dev and test have the same long e-mails), so training should match.

A fixed `max_length` with padding would waste almost all of the memory on padding. Instead each optimizer step uses 16 examples, sorted by length and cut into micro-batches of at most `MAX_TOKENS_PER_MICROBATCH` padded tokens with gradient accumulation. A step of short tickets is one forward pass; a long e-mail gets a pass of its own. The effective batch size is 16 for every step, and memory use is bounded by the longest single example.

# Part B: Baselines

In [15]:
def load_base(quant="16bit"):
    kw = dict(device_map={"": 0}, torch_dtype=torch.float16)
    if quant == "4bit":
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                                                       bnb_4bit_compute_dtype=torch.float16)
    return AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kw)

def load_for_inference(adapter_dir=None, quant="16bit"):
    """base model + (optionally) a saved PEFT adapter, loaded from disk the same way a reviewer would load it"""
    model = load_base(quant)
    if adapter_dir: model = PeftModel.from_pretrained(model, adapter_dir)
    return model.eval()

def cleanup():
    """call after `del model` to give the GPU memory back"""
    gc.collect(); torch.cuda.empty_cache()

@torch.no_grad()
def generate(model, rows, batch_size=1, system=None, **extra):
    """Inference as specified: system + user through the chat template, plain greedy model.generate, max_new_tokens=200,
    output = decoded new tokens with whitespace stripped. Nothing else touches the text."""
    prompts = [prompt_text(r["messages"], system) for r in rows]
    order = sorted(range(len(rows)), key=lambda i: len(prompts[i]))
    outs = [None] * len(rows); i = 0
    while i < len(order):
        n_tok = len(tok(prompts[order[min(i + batch_size, len(order)) - 1]], add_special_tokens=False)["input_ids"])
        idx = order[i:i + max(1, min(batch_size, 24000 // n_tok))]          # fewer rows per batch for the long e-mails
        enc = tok([prompts[j] for j in idx], return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        gen = model.generate(**enc, do_sample=False, max_new_tokens=200, **extra)
        for k, j in enumerate(idx):
            outs[j] = tok.decode(gen[k, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()
        i += len(idx)
    return [{"id": r["id"], "output": o} for r, o in zip(rows, outs)]

def dev_metrics(preds):
    res = score(dev_rows, preds)["all"]
    return {"dev_exact": res["exact_match"], "dev_field_acc": res["mean_field_acc"], "json_valid": res["json_valid"], "fields": res["fields"]}

RESULTS_PATH = f"{WORK_DIR}/experiments_log.jsonl"
def results():
    return {r["name"]: r for r in load_jsonl(RESULTS_PATH)} if os.path.exists(RESULTS_PATH) else {}
def record(row):
    with open(RESULTS_PATH, "a", encoding="utf-8") as f: f.write(json.dumps(row) + "\n")
os.makedirs(f"{WORK_DIR}/preds", exist_ok=True)

In [16]:
RULES_SYSTEM = dev_rows[0]["messages"][0]["content"] + "\n\nFollow this labelling spec exactly. Reply with only the JSON object.\n\n" + SCHEMA_TEXT

if RUN_BASELINES and not {"baseline_fixed_prompt", "baseline_rules_in_prompt"} <= set(results()):
    model = load_for_inference()
    for name, system in [("baseline_fixed_prompt", None), ("baseline_rules_in_prompt", RULES_SYSTEM)]:
        if name in results(): continue
        t0 = time.time()
        preds = generate(model, dev_rows, batch_size=GEN_BATCH, system=system)
        save_jsonl(preds, f"{WORK_DIR}/preds/dev_{name}.jsonl")
        record({"name": name, "changed": "no fine-tuning" + (", SCHEMA.md in the system prompt" if system else ""), **dev_metrics(preds),
                "gen_minutes": (time.time() - t0) / 60})
    del model; cleanup()

for name in ["baseline_fixed_prompt", "baseline_rules_in_prompt"]:
    if name in results():
        r = results()[name]
        print(f"{name:26} exact {r['dev_exact']:.1%} | mean field acc {r['dev_field_acc']:.1%} | valid JSON {r['json_valid']:.1%}")
        print("    per field:", {k: round(v, 3) for k, v in r["fields"].items()})
        print("    sample output:", repr(load_jsonl(f"{WORK_DIR}/preds/dev_{name}.jsonl")[0]["output"][:200]) if os.path.exists(f"{WORK_DIR}/preds/dev_{name}.jsonl") else "")

baseline_fixed_prompt      exact 0.0% | mean field acc 0.0% | valid JSON 0.0%
    per field: {'category': 0.0, 'priority': 0.0, 'amount_paise': 0.0, 'txn_date': 0.0, 'txn_id': 0.0, 'channel': 0.0, 'language': 0.0, 'needs_human': 0.0}
    sample output: '```json\n{\n  "status": "active",\n  "priority": "high",\n  "ticket_number": "#54047",\n  "source": "app-chat",\n  "message": "The user has received an activation guide for their card. They need assistance '
baseline_rules_in_prompt   exact 0.5% | mean field acc 37.7% | valid JSON 62.5%
    per field: {'category': 0.425, 'priority': 0.225, 'amount_paise': 0.515, 'txn_date': 0.445, 'txn_id': 0.44, 'channel': 0.42, 'language': 0.4, 'needs_human': 0.145}
    sample output: '{\n  "category": "kyc",\n  "priority": "P3",\n  "amount_paise": null,\n  "txn_date": null,\n  "txn_id": null,\n  "channel": null,\n  "language": "en",\n  "needs_human": true\n}'


The first baseline shows what the base model does with only the fixed prompt (it does not know the schema, so this is the floor). The second puts the whole spec in the system prompt. That is not allowed for the final model, but it shows how far the base model gets when it can *read* the rules instead of having learned them.

# Part C (second half): training loop

In [17]:
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

def make_steps(lengths, epochs, seed):
    """each step = EFFECTIVE_BATCH example indices of similar length (less padding); new shuffle every epoch"""
    rng = random.Random(seed); steps = []
    for _ in range(epochs):
        idx = list(range(len(lengths))); rng.shuffle(idx)
        pools = [sorted(idx[i:i + EFFECTIVE_BATCH * 8], key=lambda j: lengths[j]) for i in range(0, len(idx), EFFECTIVE_BATCH * 8)]
        ep = [p[i:i + EFFECTIVE_BATCH] for p in pools for i in range(0, len(p), EFFECTIVE_BATCH)]
        rng.shuffle(ep); steps += ep
    return steps

def micro_batches(step, lengths):
    """cut one step into forward passes of at most MAX_TOKENS_PER_MICROBATCH padded tokens"""
    out, cur = [], []
    for j in step:
        if cur and max(lengths[k] for k in cur + [j]) * (len(cur) + 1) > MAX_TOKENS_PER_MICROBATCH: out.append(cur); cur = []
        cur.append(j)
    return out + [cur]

def collate(examples, idx, device):
    L = max(len(examples[i]["input_ids"]) for i in idx)
    ids = torch.full((len(idx), L), tok.pad_token_id, dtype=torch.long)
    lab = torch.full((len(idx), L), -100, dtype=torch.long)
    att = torch.zeros((len(idx), L), dtype=torch.long)
    for b, i in enumerate(idx):
        n = len(examples[i]["input_ids"])
        ids[b, :n] = torch.tensor(examples[i]["input_ids"]); lab[b, :n] = torch.tensor(examples[i]["labels"]); att[b, :n] = 1
    return ids.to(device), att.to(device), lab.to(device)

def example_losses(model, ids, att, lab):
    """mean cross-entropy over the answer tokens, one value per example"""
    logits = model(input_ids=ids, attention_mask=att).logits
    mask = lab[:, 1:] != -100                                   # position t predicts token t+1
    tok_loss = F.cross_entropy(logits[:, :-1][mask].float(), lab[:, 1:][mask], reduction="none")
    per_example = torch.zeros(ids.shape[0], device=ids.device).index_add_(0, mask.nonzero()[:, 0], tok_loss)
    return per_example / mask.sum(1)

def train_run(name, rows, quant="16bit", r=16, lr=2e-4, epochs=3, seed=SEED):
    """LoRA fine-tune on `rows`, save the adapter (PEFT format) to WORK_DIR/runs/<name>, return run statistics"""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    examples = [encode(x["messages"]) for x in rows]
    lengths = [len(e["input_ids"]) for e in examples]
    assert max(lengths) <= MAX_TOKENS_PER_MICROBATCH, "an example is longer than one micro-batch"
    torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

    model = load_base(quant)
    if quant == "4bit":
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    else:
        model.gradient_checkpointing_enable(); model.enable_input_require_grads()
    model.config.use_cache = False
    model = get_peft_model(model, LoraConfig(r=r, lora_alpha=2 * r, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM", target_modules=LORA_TARGETS))
    params = [p for p in model.parameters() if p.requires_grad]
    for p in params: p.data = p.data.float()                    # adapter weights in fp32, frozen base in fp16 / 4-bit
    n_trainable = sum(p.numel() for p in params)
    assert n_trainable <= 50_000_000, f"{n_trainable:,} trainable parameters is over the 50M limit"

    steps = make_steps(lengths, epochs, seed)
    warmup = max(1, int(0.05 * len(steps)))
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.0)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warmup if s < warmup else 0.5 * (1 + math.cos(math.pi * (s - warmup) / max(1, len(steps) - warmup))))
    scaler = torch.amp.GradScaler("cuda")
    print(f"[{name}] {len(rows)} rows | {quant} | r={r} | lr={lr} | {epochs} epochs | {len(steps)} steps | trainable params {n_trainable:,}")

    model.train(); t0 = time.time(); curve = []
    for s, step in enumerate(steps):
        step_loss = 0.0
        for mb in micro_batches(step, lengths):
            ids, att, lab = collate(examples, mb, model.device)
            with torch.autocast("cuda", dtype=torch.float16):
                per_example = example_losses(model, ids, att, lab)
            loss = per_example.sum() / len(step)                # every example weighs the same, whatever its micro-batch
            scaler.scale(loss).backward(); step_loss += loss.item()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        curve.append(step_loss)
        if s % 25 == 0 or s == len(steps) - 1:
            print(f"  step {s + 1:4d}/{len(steps)}  loss {np.mean(curve[-25:]):.4f}  lr {sched.get_last_lr()[0]:.2e}  {(time.time() - t0) / 60:.1f} min")
    stats = {"train_minutes": (time.time() - t0) / 60, "peak_gpu_gb": torch.cuda.max_memory_allocated() / 2**30,
             "trainable_params": n_trainable, "final_loss": float(np.mean(curve[-25:])), "n_train": len(rows), "steps": len(steps)}
    out_dir = f"{WORK_DIR}/runs/{name}"
    model.save_pretrained(out_dir)
    del model, opt, sched, scaler, params; cleanup()
    return stats

**Why these settings**

| Setting | Value | Reason |
|---|---|---|
| LoRA targets | all attention + MLP projections | The task is applying rules (thresholds, date arithmetic, unit conversion), not just a change of output style. The MLP layers are where most of that capacity is; attention-only LoRA is the usual first choice for style tasks. |
| Rank / alpha | r = 16, alpha = 32 | About 18.5M trainable parameters, well under the 50M limit; the adapter file is ~74 MB, under GitHub's 100 MB limit. Rank is one of my experiments (4 / 16 / 32). Alpha = 2r keeps the update scale the same across ranks so the rank comparison is fair. |
| Learning rate | 2e-4, 5% warm-up then cosine to 0 | Standard LoRA starting point. Warm-up because the B matrices start at zero and the first steps are noisy; cosine so the last steps are small. |
| Epochs | 3 | About 1,300 to 1,550 rows; the answers are short and highly regular, so the loss flattens quickly (see the printed curve). More epochs mainly memorise ticket wording. |
| Effective batch | 16 | Enough steps (≈ 80 to 100 per epoch) for the schedule to matter, each step still averaging over all seven categories. |
| Precision | fp16 base, fp32 adapter, `autocast` + `GradScaler` | T4 has no fast bf16. Keeping the trained weights in fp32 avoids fp16 underflow in the optimizer update. |
| Gradient checkpointing | on | Makes the 2,000+ token e-mails fit on a T4 in both the 16-bit and 4-bit runs, so both use identical batches. |
| Seed | fixed | Same data order and LoRA init across experiments, so only the stated change differs. |

# Part D: Controlled experiments

`main` is the reference. Every other run changes exactly one thing relative to `main`.

In [18]:
RAW_NO_LEAK = [r for r in train_raw if norm_key(U(r)) not in dev_keys]     # raw labels as delivered, minus rows that are dev tickets

EXPERIMENTS = {   # name: (what changed vs main, training rows, train_run arguments)
    "main":       ("reference: cleaned data, 16-bit LoRA, r=16, lr 2e-4, 3 epochs", train_clean, {}),
    "raw_data":   ("data: raw train.jsonl instead of cleaned (dev-leak rows removed, see note)", RAW_NO_LEAK, {}),
    "qlora_4bit": ("precision: 4-bit QLoRA (NF4, double quant) instead of 16-bit", train_clean, {"quant": "4bit"}),
    "rank_4":     ("rank: r=4 (alpha 8) instead of r=16", train_clean, {"r": 4}),
    "rank_32":    ("rank: r=32 (alpha 64) instead of r=16", train_clean, {"r": 32}),
    "clean_aug":  ("data: cleaned + spec-driven augmentation (own idea)", train_clean + train_aug, {}),
}
assert FINAL in EXPERIMENTS

def run_experiment(name):
    changed, rows, kw = EXPERIMENTS[name]
    stats = train_run(name, rows, **kw)
    model = load_for_inference(f"{WORK_DIR}/runs/{name}", kw.get("quant", "16bit"))      # evaluated in the precision it was trained in
    preds = generate(model, dev_rows, batch_size=GEN_BATCH)
    del model; cleanup()
    save_jsonl(preds, f"{WORK_DIR}/preds/dev_{name}.jsonl")
    record({"name": name, "changed": changed, **dev_metrics(preds), **stats, **{"quant": "16bit", "r": 16, "lr": 2e-4, "epochs": 3, **kw}})

for name in EXPERIMENTS:
    have = name in results() and (name != FINAL or os.path.exists(f"{WORK_DIR}/runs/{name}/adapter_config.json"))
    if (RUN_ALL_EXPERIMENTS or name == FINAL) and not have:
        run_experiment(name)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[main] 1267 rows | 16bit | r=16 | lr=0.0002 | 3 epochs | 240 steps | trainable params 18,464,768
  step    1/240  loss 1.6315  lr 3.33e-05  0.2 min
  step   26/240  loss 0.5204  lr 1.98e-04  1.8 min
  step   51/240  loss 0.0408  lr 1.86e-04  3.3 min
  step   76/240  loss 0.0284  lr 1.64e-04  4.5 min
  step  101/240  loss 0.0194  lr 1.34e-04  5.9 min
  step  126/240  loss 0.0112  lr 1.00e-04  7.6 min
  step  151/240  loss 0.0072  lr 6.62e-05  9.0 min
  step  176/240  loss 0.0045  lr 3.64e-05  10.4 min
  step  201/240  loss 0.0023  lr 1.41e-05  12.1 min
  step  226/240  loss 0.0016  lr 1.85e-06  13.3 min
  step  240/240  loss 0.0014  lr 0.00e+00  14.2 min


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

[rank_32] 1267 rows | 16bit | r=32 | lr=0.0002 | 3 epochs | 240 steps | trainable params 36,929,536
  step    1/240  loss 1.6315  lr 3.33e-05  0.2 min
  step   26/240  loss 0.4080  lr 1.98e-04  1.8 min
  step   51/240  loss 0.0383  lr 1.86e-04  3.3 min
  step   76/240  loss 0.0246  lr 1.64e-04  4.5 min
  step  101/240  loss 0.0117  lr 1.34e-04  6.0 min
  step  126/240  loss 0.0058  lr 1.00e-04  7.6 min
  step  151/240  loss 0.0038  lr 6.62e-05  9.0 min
  step  176/240  loss 0.0028  lr 3.64e-05  10.5 min
  step  201/240  loss 0.0012  lr 1.41e-05  12.1 min
  step  226/240  loss 0.0011  lr 1.85e-06  13.3 min
  step  240/240  loss 0.0007  lr 0.00e+00  14.2 min


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[clean_aug] 1559 rows | 16bit | r=16 | lr=0.0002 | 3 epochs | 294 steps | trainable params 18,464,768
  step    1/294  loss 2.0877  lr 2.86e-05  0.0 min
  step   26/294  loss 0.5894  lr 1.99e-04  1.8 min
  step   51/294  loss 0.0418  lr 1.92e-04  3.1 min
  step   76/294  loss 0.0314  lr 1.77e-04  4.7 min
  step  101/294  loss 0.0233  lr 1.56e-04  5.8 min
  step  126/294  loss 0.0099  lr 1.31e-04  7.0 min
  step  151/294  loss 0.0072  lr 1.03e-04  8.9 min
  step  176/294  loss 0.0041  lr 7.56e-05  10.3 min
  step  201/294  loss 0.0038  lr 4.97e-05  11.7 min
  step  226/294  loss 0.0024  lr 2.77e-05  13.1 min
  step  251/294  loss 0.0023  lr 1.14e-05  14.2 min
  step  276/294  loss 0.0012  lr 2.03e-06  15.8 min
  step  294/294  loss 0.0016  lr 0.00e+00  16.8 min


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [19]:
def experiment_table():
    res = results()
    lines = ["| Run | What changed | Train rows | Dev exact match | Dev mean field acc | Peak GPU (GB) | Train (min) |", "|---|---|---|---|---|---|---|"]
    for name in ["baseline_fixed_prompt", "baseline_rules_in_prompt"] + list(EXPERIMENTS):
        if name not in res: lines.append(f"| {name} | not run in this session | | | | | |"); continue
        r = res[name]; num = lambda k, f: format(r[k], f) if k in r else "n/a"
        lines.append(f"| {name}{' **(final)**' if name == FINAL else ''} | {r['changed']} | {r.get('n_train', 'n/a')} | {r['dev_exact']:.1%} | {r['dev_field_acc']:.1%} | {num('peak_gpu_gb', '.1f')} | {num('train_minutes', '.1f')} |")
    return "\n".join(lines)

table_md = experiment_table()
open(f"{WORK_DIR}/experiment_table.md", "w").write(table_md)
try:
    from IPython.display import Markdown, display
    display(Markdown(table_md))
except Exception:
    print(table_md)
print("\nper-field dev accuracy:")
for name, r in results().items(): print(f"  {name:26}", " ".join(f"{k[:9]}={v:.2f}" for k, v in r["fields"].items()))

| Run | What changed | Train rows | Dev exact match | Dev mean field acc | Peak GPU (GB) | Train (min) |
|---|---|---|---|---|---|---|
| baseline_fixed_prompt | no fine-tuning | n/a | 0.0% | 0.0% | n/a | n/a |
| baseline_rules_in_prompt | no fine-tuning, SCHEMA.md in the system prompt | n/a | 0.5% | 37.7% | n/a | n/a |
| main | reference: cleaned data, 16-bit LoRA, r=16, lr 2e-4, 3 epochs | 1267 | 87.5% | 97.8% | 6.0 | 14.2 |
| raw_data | data: raw train.jsonl instead of cleaned (dev-leak rows removed, see note) | 1400 | 78.5% | 96.6% | 5.9 | 15.7 |
| qlora_4bit | precision: 4-bit QLoRA (NF4, double quant) instead of 16-bit | 1267 | 86.5% | 97.4% | 5.4 | 17.1 |
| rank_4 | rank: r=4 (alpha 8) instead of r=16 | 1267 | 63.0% | 93.2% | 5.8 | 14.3 |
| rank_32 | rank: r=32 (alpha 64) instead of r=16 | 1267 | 92.0% | 98.4% | 6.2 | 14.2 |
| clean_aug **(final)** | data: cleaned + spec-driven augmentation (own idea) | 1559 | 89.5% | 98.0% | 6.0 | 16.8 |


per-field dev accuracy:
  baseline_fixed_prompt      category=0.00 priority=0.00 amount_pa=0.00 txn_date=0.00 txn_id=0.00 channel=0.00 language=0.00 needs_hum=0.00
  baseline_rules_in_prompt   category=0.42 priority=0.23 amount_pa=0.52 txn_date=0.45 txn_id=0.44 channel=0.42 language=0.40 needs_hum=0.14
  main                       category=0.96 priority=0.91 amount_pa=0.99 txn_date=0.98 txn_id=0.99 channel=1.00 language=1.00 needs_hum=0.97
  raw_data                   category=0.97 priority=0.83 amount_pa=0.98 txn_date=0.99 txn_id=1.00 channel=0.99 language=1.00 needs_hum=0.95
  qlora_4bit                 category=0.98 priority=0.89 amount_pa=1.00 txn_date=0.98 txn_id=1.00 channel=1.00 language=1.00 needs_hum=0.94
  rank_4                     category=0.96 priority=0.69 amount_pa=0.98 txn_date=0.96 txn_id=1.00 channel=0.99 language=1.00 needs_hum=0.88
  rank_32                    category=0.98 priority=0.94 amount_pa=0.99 txn_date=0.99 txn_id=1.00 channel=1.00 language=1.00 needs_hum=

Notes on reading the table

* **`raw_data`**: the assignment forbids training on dev rows, and 45 raw train rows are dev tickets. I removed those from the raw run too, otherwise its dev score would be inflated by memorised answers. Everything else in that run is the data as delivered: truncated labels, old label format, rupee amounts, wrong priorities, duplicates.
* Experiment dev scores use batched generation (batch 16) for speed. The final model below is scored again one ticket at a time, which is the number I report as final.
* Each run is a single seed. Dev has 200 rows, so one ticket is 0.5 points of exact match. I treat differences of one or two tickets as noise.

# Final model: dev score and test predictions

The final adapter is the `FINAL` run from the table. It is loaded from disk the way a reviewer would load it (base model + `PeftModel.from_pretrained`) and run one ticket at a time with plain greedy `model.generate`.

In [20]:
SUBMIT_DIR = f"{WORK_DIR}/submission"
final_quant = EXPERIMENTS[FINAL][2].get("quant", "16bit")
os.makedirs(SUBMIT_DIR, exist_ok=True)
shutil.rmtree(f"{SUBMIT_DIR}/adapter", ignore_errors=True)
shutil.copytree(f"{WORK_DIR}/runs/{FINAL}", f"{SUBMIT_DIR}/adapter", ignore=shutil.ignore_patterns("README.md", "checkpoint*"))
print("adapter files:", {f: f"{os.path.getsize(f'{SUBMIT_DIR}/adapter/{f}') / 2**20:.1f} MB" for f in os.listdir(f"{SUBMIT_DIR}/adapter")})

model = load_for_inference(f"{SUBMIT_DIR}/adapter", final_quant)
t0 = time.time()
final_dev = generate(model, dev_rows, batch_size=1)
save_jsonl(final_dev, f"{SUBMIT_DIR}/predictions_dev.jsonl")
final_res = score(dev_rows, final_dev)
m = final_res["all"]
print(f"FINAL ({FINAL}, base loaded in {final_quant}) on dev, batch size 1: exact match {m['exact_match']:.1%} | mean field accuracy {m['mean_field_acc']:.1%} | valid JSON {m['json_valid']:.1%} | {(time.time() - t0) / 60:.1f} min")

# does batching change any output? (experiments used batch 16)
batched = {p["id"]: p["output"] for p in load_jsonl(f"{WORK_DIR}/preds/dev_{FINAL}.jsonl")} if os.path.exists(f"{WORK_DIR}/preds/dev_{FINAL}.jsonl") else {}
if batched: print("dev outputs that differ between batch 16 and batch 1:", sum(batched[p["id"]] != p["output"] for p in final_dev), "of", len(final_dev))

# The base model's generation_config.json (printed in Part C) sets a repetition_penalty, and plain model.generate() applies it.
# How many dev outputs would change if a reviewer's script switches it off?
no_pen = generate(model, dev_rows, batch_size=1, repetition_penalty=1.0)
changed = [(a["id"], a["output"], b["output"]) for a, b in zip(final_dev, no_pen) if a["output"] != b["output"]]
m2 = score(dev_rows, no_pen)["all"]
print(f"with repetition_penalty=1.0: {len(changed)} of {len(final_dev)} dev outputs differ | exact match {m2['exact_match']:.1%} | mean field accuracy {m2['mean_field_acc']:.1%}")
for c in changed[:5]: print("  ", c)

adapter files: {'adapter_model.safetensors': '70.5 MB', 'adapter_config.json': '0.0 MB'}


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

FINAL (clean_aug, base loaded in 16bit) on dev, batch size 1: exact match 89.5% | mean field accuracy 98.0% | valid JSON 100.0% | 18.4 min
dev outputs that differ between batch 16 and batch 1: 0 of 200
with repetition_penalty=1.0: 6 of 200 dev outputs differ | exact match 92.5% | mean field accuracy 98.5%
   ('dv-00034', '{"category": "payment_failed", "priority": "P1", "amount_paise": 3673700, "txn_date": null, "txn_id": "NP9442425625", "channel": "wallet", "language": "hi-en", "needs_human": true}', '{"category": "payment_failed", "priority": "P2", "amount_paise": 3673700, "txn_date": null, "txn_id": "NP9442425625", "channel": "wallet", "language": "hi-en", "needs_human": false}')
   ('dv-00045', '{"category": "payment_failed", "priority": "P3", "amount_paise": null, "txn_date": "2025-05-27", "txn_id": "NP6878764609", "channel": "card", "language": "en", "needs_human": false}', '{"category": "payment_failed", "priority": "P3", "amount_paise": null, "txn_date": "2026-05-27", "txn_id":

In [21]:
t0 = time.time()
test_preds = generate(model, test_rows, batch_size=1)
save_jsonl(test_preds, f"{SUBMIT_DIR}/predictions_test.jsonl")
del model; cleanup()

def parses(o):
    try: x = json.loads(o); return isinstance(x, dict) and set(x) == set(KEYS)
    except Exception: return False
print(f"test predictions: {len(test_preds)} rows in {(time.time() - t0) / 60:.1f} min | ids match test file: {[p['id'] for p in test_preds] == [r['id'] for r in test_rows]}"
      f" | valid JSON with the 8 keys: {sum(parses(p['output']) for p in test_preds)}")
print("predicted category mix on test:", dict(Counter(json.loads(p["output"])["category"] for p in test_preds if parses(p["output"]))))
for p in test_preds[:3]: print("  ", p)

shutil.copy(f"{WORK_DIR}/cleaning_log.csv", SUBMIT_DIR); shutil.copy(RESULTS_PATH, SUBMIT_DIR); shutil.copy(f"{WORK_DIR}/experiment_table.md", SUBMIT_DIR)
print("\nfiles for the repository are in", SUBMIT_DIR, "->", sorted(os.listdir(SUBMIT_DIR)))
print(f"\nREADME line: {BASE_MODEL}, base loaded in {'4-bit (bitsandbytes NF4)' if final_quant == '4bit' else '16-bit fp16 (not 4-bit)'}; "
      f"final run '{FINAL}': dev exact match {m['exact_match']:.1%}, mean field accuracy {m['mean_field_acc']:.1%}.")

test predictions: 400 rows in 36.4 min | ids match test file: True | valid JSON with the 8 keys: 400
predicted category mix on test: {'fraud': 16, 'offers': 46, 'payment_failed': 135, 'refund': 89, 'kyc': 41, 'other': 27, 'account_access': 46}
   {'id': 'ts-00001', 'output': '{"category": "fraud", "priority": "P1", "amount_paise": null, "txn_date": "2026-07-00", "txn_id": "NP6090913363", "channel": null, "language": "hi-en", "needs_human": true}'}
   {'id': 'ts-00002', 'output': '{"category": "offers", "priority": "P3", "amount_paise": 1302700, "txn_date": "2026-07-29", "txn_id": null, "channel": "card", "language": "hi-en", "needs_human": false}'}
   {'id': 'ts-00003', 'output': '{"category": "payment_failed", "priority": "P1", "amount_paise": 61875, "txn_date": "2026-09-11", "txn_id": null, "channel": null, "language": "en", "needs_human": true}'}

files for the repository are in /content/drive/MyDrive/nimbuspay_triage/submission -> ['adapter', 'cleaning_log.csv', 'experiment_table.m

# Part E: Error analysis on dev (final model)

In [22]:
print(f"{'group':28} {'n':>4} {'exact':>7} {'fields':>7}")
for name in sorted(final_res, key=lambda x: (x != "all", x)):
    r = final_res[name]; print(f"{name:28} {r['n']:4d} {r['exact_match']:7.1%} {r['mean_field_acc']:7.1%}")
print("\nper-field accuracy (all dev rows):")
for k, v in sorted(final_res["all"]["fields"].items(), key=lambda kv: kv[1]): print(f"  {k:14} {v:6.1%}")

# slices by what the ticket contains, to see which rule the errors come from
SLICES = {
    "hinglish": lambda t, g, r: g["language"] == "hi-en",
    "long e-mail with quoted thread": lambda t, g, r: "\n>" in U(r),
    "relative date (kal/parso/yesterday/N days ago)": lambda t, g, r: re.search(r"\bkal\b|\bparso\b|yesterday|today|\baaj\b|days? ago|din pehle", t, re.I),
    "date without a year": lambda t, g, r: g["txn_date"] and not re.search(r"20\d\d|\d\d[-/]\d\d[-/]\d\d", t) and not re.search(r"\bkal\b|\bparso\b|yesterday|today|\baaj\b|days? ago|din pehle", t, re.I),
    "amount with k / lakh": lambda t, g, r: g["amount_paise"] and re.search(r"\d\s?k\b|lakh|\blacs?\b", t, re.I),
    "amount with a decimal part": lambda t, g, r: g["amount_paise"] and g["amount_paise"] % 100,
    "second amount (balance/limit/fee)": lambda t, g, r: re.search(r"balance|limit|\bfees?\b|pade hain", t, re.I),
    "txn id not already normalised in the text": lambda t, g, r: g["txn_id"] and g["txn_id"] not in t,
    "other reference number present": lambda t, g, r: re.search(r"#\d+|bank ref|order (no|number|id)", t, re.I),
    "legal threat": lambda t, g, r: LEGAL.search(t),
    "contacted before": lambda t, g, r: REPEAT.search(t),
    "amount >= Rs 50,000": lambda t, g, r: g["amount_paise"] and g["amount_paise"] >= 5000000,
}
pred_by_id = {p["id"]: p["output"] for p in final_dev}
def parse_pred(s):
    try: x = json.loads(s); return x if isinstance(x, dict) else None
    except Exception: return None
rows_e = []
for r in dev_rows:
    g = json.loads(A(r)); p = parse_pred(pred_by_id[r["id"]])
    wrong = [k for k in KEYS if p is None or k not in p or type(p[k]) is not type(g[k]) or p[k] != g[k]]
    rows_e.append((r, g, p, wrong))
print(f"\n{'slice':48} {'n':>4} {'exact':>7}   wrong fields")
for name, f in SLICES.items():
    sel = [x for x in rows_e if f(split_ticket(U(x[0]))[2], x[1], x[0])]
    if sel: print(f"{name:48} {len(sel):4d} {np.mean([not x[3] for x in sel]):7.1%}   {dict(Counter(k for x in sel for k in x[3]))}")

group                           n   exact  fields
all                           200   89.5%   98.0%
category=account_access        19   94.7%   99.3%
category=fraud                 15   86.7%   95.0%
category=kyc                   24  100.0%  100.0%
category=offers                18  100.0%  100.0%
category=other                 16  100.0%  100.0%
category=payment_failed        53   81.1%   96.7%
category=refund                55   85.5%   97.5%

per-field accuracy (all dev rows):
  priority        92.0%
  needs_human     96.0%
  category        98.0%
  amount_paise    98.5%
  txn_date        99.5%
  txn_id         100.0%
  channel        100.0%
  language       100.0%

slice                                               n   exact   wrong fields
hinglish                                           49   87.8%   {'priority': 6, 'needs_human': 2, 'category': 1, 'amount_paise': 1}
long e-mail with quoted thread                      3   33.3%   {'priority': 1, 'needs_human': 1}
relative date 

In [23]:
fails = [x for x in rows_e if x[3]]
print(f"{len(fails)} dev tickets with at least one wrong field. All of them:\n")
for r, g, p, wrong in fails:
    rec, src, t = split_ticket(U(r))
    print(f"{r['id']}  [{g['category']}, {src}, received {rec} {rec.strftime('%a')}]  wrong: {wrong}")
    print("   ticket:", t.strip().replace("\n", " | ")[:420])
    if p is None: print("   output is not valid JSON:", repr(pred_by_id[r["id"]][:200]))
    else:
        for k in wrong: print(f"   {k}: predicted {p.get(k)!r}   gold {g[k]!r}")
    print()

21 dev tickets with at least one wrong field. All of them:

dv-00010  [payment_failed, email, received 2026-07-09 Thu]  wrong: ['category', 'priority', 'needs_human']
   ticket: Subject: Problem with NimbusPay |  | URGENT!! | I didn't see the money I added to my card. ID: NP0194178721 INR 2,570 was the value. Please look into it. |  | Regards, | Vikram Singh
   category: predicted 'fraud'   gold 'payment_failed'
   priority: predicted 'P1'   gold 'P3'
   needs_human: predicted True   gold False

dv-00017  [refund, twitter, received 2026-06-30 Tue]  wrong: ['amount_paise']
   ticket: @NimbusPayCare I am still waiting for a refund. Mode: internet banking. My account balance is 712 rupees right now. It took place on 13th June 2026. You can check np5682637619. The amount was Rs 1,80,827.25.
   amount_paise: predicted 18082525   gold 18082725

dv-00018  [refund, app-chat, received 2026-07-31 Fri]  wrong: ['priority']
   ticket: Sir, Order cnacel kar diya tha par refnud abhi tak nahi aaya. A

My reading of these failures, the ten I picked with an explanation each, and what I would do with one more week are in `report.md` (Part E).